NEW CPU model

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score
import joblib

# Define file paths
attack_file = "/content/drive/MyDrive/Dataset/attack/all_dns.csv"
normal_file = "/content/drive/MyDrive/Dataset/normal/merged_dns.csv"


In [ ]:

# 1) Load datasets using pandas (CPU-based)
attack_df = pd.read_csv(attack_file)
normal_df = pd.read_csv(normal_file)

# 2) Combine datasets
data = pd.concat([attack_df, normal_df], ignore_index=True)

# 3) Use all available columns as features except 'category'
feature_columns = [col for col in data.columns if col != 'category']

# Ensure required columns exist (fills missing with 0)
def ensure_columns(df, columns):
    missing_cols = [col for col in columns if col not in df.columns]
    for col in missing_cols:
        df[col] = 0  # Fill missing columns with default values

ensure_columns(attack_df, feature_columns)
ensure_columns(normal_df, feature_columns)


<ipython-input-9-24b76cb65700>:2: DtypeWarning: Columns (9,11,13) have mixed types. Specify dtype option on import or set low_memory=False.
  attack_df = pd.read_csv(attack_file)


In [ ]:

# 4) (Optional) Re-check combined 'data' if needed — it's already combined above.

# 5) Encode categorical columns (CPU-based) using scikit-learn
label_encoders = {}
for col in feature_columns:
    if data[col].dtype == 'object':
        data[col] = data[col].astype(str).fillna("unknown")
        le = LabelEncoder()
        data[col] = le.fit_transform(data[col])
        label_encoders[col] = le


In [ ]:

# 6) Extract features (X) and target (y)
X = data[feature_columns]
y = data['category'].astype(str).fillna("unknown")

# Encode the target labels
y_le = LabelEncoder()
y = y_le.fit_transform(y)

# 7) Split data into train (60%), test (20%), and validation (20%)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=42)
X_test, X_val, y_test, y_val = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)


In [ ]:

# 8) Train CPU-based Random Forest model from scikit-learn
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

# 9) Make predictions (test set)
y_pred = clf.predict(X_test)

# 10) Evaluate model
accuracy = accuracy_score(y_test, y_pred) * 100
f1 = f1_score(y_test, y_pred, average='weighted')

print(f"Accuracy: {accuracy:.2f}%")
print(f"F1 Score: {f1:.4f}")


Accuracy: 98.62%
F1 Score: 0.9859


In [ ]:

# 11) Validate model on validation set
y_val_pred = clf.predict(X_val)
val_accuracy = accuracy_score(y_val, y_val_pred) * 100
val_f1 = f1_score(y_val, y_val_pred, average='weighted')

print(f"Validation Accuracy: {val_accuracy:.2f}%")
print(f"Validation F1 Score: {val_f1:.4f}")


Validation Accuracy: 98.64%
Validation F1 Score: 0.9861


In [ ]:

# 12) Save model and label encoder for CPU-based inference
joblib.dump(clf, "/content/drive/MyDrive/Dataset/Saved_model/CPU-Model/random_forest_cpu_model_dns.joblib")
joblib.dump(y_le, "/content/drive/MyDrive/Dataset/Saved_model/CPU-Model/label_encoder_dns.joblib")

print("Model saved as 'random_forest_cpu_model_dns.joblib'")
print("Label encoder saved as 'label_encoder_dns.joblib'")


Model saved as 'random_forest_cpu_model_dns.joblib'
Label encoder saved as 'label_encoder_dns.joblib'
